In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# Policy gradient I: improving a policy directly

## Where this session fits

> **Part A ended by proposing a network that *is* the policy. Part B builds that network and shows what training it requires.**

- **Where Part A left us**
    - its last slide proposed a network that outputs the policy
    - Part A never built one: it learned values and read the policy off them with an argmax
    - the limitation: the argmax needs a short list of actions and gives greedy (deterministic) policies
- **The purpose of Part B** (policy gradient, session 1 of 4)
    - **build the network** that outputs the policy: the probabilities of the actions
    - **train it by gradient ascent**, which needs the gradient of the expected return with respect to $\theta$
- **The obstacle:** the return depends on the environment's response, which is unknown, so we cannot differentiate through it
    - **one theorem** removes the obstacle; this session states it
    - the theorem is the common core of the methods of Part B (REINFORCE, actor-critic, PPO) and of GRPO in Part C
    - each estimates this gradient from sampled episodes, and they differ in a few choices (previewed at the end of this session)
- **Series goal:** choose good actions by trial and error, from small tables to robots and language models


**By the end you can**

1. say what a policy network outputs, for discrete and for continuous actions
2. write the objective $J(\theta)$ and say why its gradient is not obvious
3. state the policy gradient theorem in its two variants and say why the transition probabilities $\transp$ do not appear
4. say what one update does to the action probabilities
5. say why the raw return is a noisy weight, and what a **baseline** is
6. say why this estimator is on-policy
7. say where the proofs are, and why to read them


**Where this session sits in Part B**

- **this session, in three steps**
    1. the network: what a policy network outputs
    2. the gradient we need: gradient ascent on the expected return $J(\theta)$
    3. the theorem that lets us compute that gradient from sampled episodes

<img src="images/partB_arc_b1.png" width=95%>


# A policy as a network

**Why this section?** Part A ended at the limits of the argmax, so Part B learns the policy directly. First, what does a policy network output?

> **The policy network outputs a probability distribution over actions, and the agent samples its action from it.**

- the policy $\pi_\theta(\act \mid \state) = \pr{\actseq_\tt = \act \mid \stateseq_\tt = \state}$ is a *probability distribution* over actions, given the state (a *density* when actions are continuous)
    - $\theta$ are the network's parameters
- the network outputs the numbers that define the distribution
- the agent **samples** its action from it
    - **discrete actions:** one **logit** (any real number) per action, turned into probabilities by a **softmax** (exponentiate the logits and divide by their sum)
    - **continuous actions:** the network outputs the mean and spread of a Gaussian (a bell-shaped distribution), and the action is sampled from it
    - Microduck's policy is of the second kind: continuous motor targets

**Randomness is built in**

- randomness is built into the policy, so exploration needs no $\epsilon$-greedy rule (the spread may shrink as learning proceeds)

<img src="images/policy_heads.png" width=90%>


**Two sources of randomness**

- the **policy** is random: it samples $\act$ from $\pi_\theta(\cdot \mid \state)$
- the **environment** can also be random: $\transp(\state', \rew \mid \state, \act)$
    - a single observed $(\state', \rew)$ is only a noisy sample of that distribution
    - every model-free method must work from such samples
- the theorem of this session will show that the gradient never needs $\transp$ itself


## Improving the policy: gradient ascent on $J(\theta)$

**Why this section?** With a parameterized policy, "improving the policy" means changing $\theta$. That needs a number to improve and a direction to move.

- $J(\theta)$ is the **objective**: a number saying how good $\pi_\theta$ is
- $\nabla_\theta J(\theta)$ is its gradient with respect to the parameters $\theta$
- $\alpha$ is the step size (learning rate)
- policy gradient methods create a sequence of policies $\pi_0, \ldots, \pi_p, \ldots$ by a sequence of parameters $\theta_0, \ldots, \theta_p, \ldots$
- each step is **gradient ascent** on an objective $J(\theta)$
    - ascent, not descent, because we *maximize*
    - we add the gradient where a loss would subtract it

$$
\theta_{p+1} = \theta_p + \alpha \, \nabla_\theta J(\theta_p)
$$

- source for the chapter: Sutton and Barto, *Reinforcement Learning*, Chapter 13


## The objective: expected return

**Why this section?** Gradient ascent needs something to maximize: the expected return.

> **$J(\theta)$ is the expected return: the average, over episodes generated by $\pi_\theta$, of the return from the initial state.**

$$
J(\theta) = \Exp{\traj \sim \pi_\theta} ( \rlret_{0, \traj} )
$$

- $\traj \sim \pi_\theta$ means episodes generated by following $\pi_\theta$
- this is the value of the initial state of *this* policy, $\statevalfun_{\pi_\theta}(\stateseq_0)$, from Part A
    - the subscript $\pi_\theta$: the one policy with the current parameters $\theta$, not all policies
    - averaged over the distribution of starting states, if it is not fixed
- **for each fixed $\theta$, $\pi_\theta$ defines a distribution over episodes**
    - a different $\theta$ gives a different distribution, so $J$ is a function of $\theta$
    - $\theta$ is the argument of $J$ *and* sits inside the distribution we average over

**The same expectation as a sum over episodes**

> **Each episode counts in proportion to its probability $\prc{\traj}{\theta}$ under the current $\theta$: the policy $\pi_\theta$ decides which episodes are likely.**

$$
J(\theta) = \Exp{\traj \sim \pi_\theta} ( \rlret_{0, \traj} ) = \sum_\traj \prc{\traj}{\theta} \, \rlret_{0, \traj}
$$

- $\prc{\traj}{\theta}$ is the probability that following $\pi_\theta$ produces the episode $\traj$
- change $\theta$ and the weights $\prc{\traj}{\theta}$ change, so $J$ changes
- the formula for $\prc{\traj}{\theta}$ is written out on a later slide

**Reading the formula: what changes from Part A**

- **notation change:** every sequence quantity now carries its trajectory
    - the return: $\rlret_{\tt,\traj}$ (Part A: $\rlret_\tt$)
    - states, actions and rewards: $\stateseq_{\traj,\tt}$, $\actseq_{\traj,\tt}$, $\rewseq_{\traj,\tt+1}$ (Part A: $\stateseq_\tt$, $\actseq_\tt$, $\rewseq_{\tt+1}$)
    - the subscript $\traj$ says which episode the quantity belongs to
    - the formulas average over episodes, so they must name the episode

Recall the return from step $\tt$ of an episode

$$
\rlret_{\tt, \traj} = \sum_{k=0}^{|\traj|-\tt-1} \disc^k \, \rewseq_{\tt+k+1} = \rewseq_{\tt+1} + \disc \, \rlret_{\tt+1, \traj}
$$

**Three probabilities, kept apart**

- three probabilities appear in this session
    - $\transp$ (capital $P$): the environment's response, outside the agent's control
    - $\pi_\theta$: the policy, the agent's choice of action
    - $\prc{\traj}{\theta}$ (small $p$): the probability of a whole episode, which combines both
- the probability $\prc{\traj}{\theta}$ of an episode depends on $\theta$, through the policy


## Why the gradient is not obvious

**Why this section?** $J$ is an expectation over episodes, and the environment's probabilities are unknown. How can we get its gradient at all?

> **The question:** we need $\nabla_\theta J(\theta_p)$, but $J$ is an average over episodes whose probabilities involve the unknown environment. How do we differentiate it?

- recall the expectation as a probability-weighted sum over episodes (the objective slide)

$$
\Exp{\traj \sim \pi_\theta} ( \rlret_{0, \traj} ) = \sum_\traj \prc{\traj}{\theta} \, \rlret_{0, \traj}
$$


**The obstacle: an episode's probability contains the environment**

- the probability of an episode chains the policy and the environment at every step

$$
\prc{\traj}{\theta} = \pr{\stateseq_{\traj,0}} \prod_{\tt=0}^{|\traj|-1} \pi_\theta(\actseq_{\traj,\tt} \mid \stateseq_{\traj,\tt}) \; \transp(\stateseq_{\traj, \tt+1}, \rewseq_{\traj,\tt+1} \mid \stateseq_{\traj,\tt}, \actseq_{\traj,\tt})
$$

- the $\transp$ factors (and the start distribution) are the environment's response, **not available** in a model-free setting

<img src="images/b1_trajectory_chain.png" width=90%>


# The policy gradient theorem

**Why this section?** It answers the question just posed: the gradient can be written without the environment's probabilities.

> **Theorem: $\nabla_\theta J(\theta)$ is**
>
> - **an expectation** over episodes generated by $\pi_\theta$
> - **of** $\nabla_\theta \log \pi_\theta$ (how a change in $\theta$ changes the log-probability of the action taken)
> - **weighted by** a return

> **Why the theorem matters: the transition probabilities $\transp$ do not appear.**

- **conditions**
    - episodic tasks
    - $\pi_\theta$ differentiable in $\theta$
    - the start distribution and $\transp$ do not depend on $\theta$
- so the gradient can be *estimated from sampled episodes*, without knowing $\transp$

**Why the transition probabilities disappear**

- **the probability of an episode is a product**, with one factor per step (previous slide)
    - the policy picks the action: $\pi_\theta(\actseq_{\tau,\tt} \mid \stateseq_{\tau,\tt})$, which depends on $\theta$
    - the environment responds to that action: $\transp(\stateseq_{\tau,\tt+1}, \rewseq_{\tau,\tt+1} \mid \stateseq_{\tau,\tt}, \actseq_{\tau,\tt})$, which does **not** depend on $\theta$
    - the start distribution does not depend on $\theta$ either
- a product is awkward to differentiate, so take the log: **the log of a product is a sum**

$$
\log \prc{\tau}{\theta} = \log \pr{\stateseq_{\tau,0}} + \sum_{\tt=0}^{|\tau|-1} \Big[ \log \pi_\theta(\actseq_{\tau,\tt} \mid \stateseq_{\tau,\tt}) + \log \transp(\stateseq_{\tau,\tt+1}, \rewseq_{\tau,\tt+1} \mid \stateseq_{\tau,\tt}, \actseq_{\tau,\tt}) \Big]
$$

- the gradient of a sum is the sum of the gradients
- terms without $\theta$ have gradient $0$: the start and $\transp$ terms drop out
- only $\sum_{\tt} \nabla_\theta \log \pi_\theta$ remains (details: the appendix)

**Two variants of the theorem**

**Episode-return variant**: every step's term is weighted by the whole episode's return

$$
\nabla_\theta J(\theta) = \Exp{\traj \sim \pi_\theta} \left[ \sum_{\tt=0}^{|\traj|-1} \nabla_\theta \log \pi_\theta(\actseq_{\traj,\tt} \mid \stateseq_{\traj,\tt}) \; \rlret_{0,\traj} \right]
$$

- natural when the reward comes at the end of the episode (a single reward)


**Return-to-go variant**: each step's term is weighted only by the return from that step on

$$
\nabla_\theta J(\theta) = \E_{\traj \sim \pi_\theta} \left[ \sum_{\tt=0}^{|\traj|-1} \nabla_\theta \log \pi_\theta(\actseq_{\traj,\tt} \mid \stateseq_{\traj,\tt}) \; \disc^\tt \, \rlret_{\tt,\traj} \right]
$$

- natural when rewards arrive at intermediate steps
- an action cannot affect rewards that came before it
- the two variants are equal *in expectation* (path by path they differ, except for a single terminal reward)
- the factor $\disc^\tt$ is $1$ when $\disc = 1$
- many implementations drop $\disc^\tt$
- the gradient they compute is then a different (biased) estimate for the discounted objective


**What the theorem gives us, either way**

- this theorem is the foundation of the stochastic-policy gradient methods of this part (deterministic-policy methods rest on a different result)
- $\nabla_\theta \log \pi_\theta(\act \mid \state)$ (called the **score function**, or just the *score*) is the gradient of the log probability of the action taken
    - it is the direction in parameter space that makes action $\act$ more likely in state $\state$
- the weight in front of it (here a return) says *how much* to push


**What one update does** (one state, a softmax policy over three actions)

> **An action followed by a positive return becomes more likely. One followed by a negative return, less likely.**

- the setting: one state, three actions
    - the parameters $\theta$ are the three *logits*, here $\theta = (0.4, 0, -0.4)$
    - so $\pi_\theta \approx (0.47, 0.32, 0.21)$
    - the second action is the one sampled
- the update: $\theta \leftarrow \theta + \alpha \, \rlret \, \nabla_\theta \log \pi_\theta(\act)$
    - $\rlret$ is the return that followed the sampled action $\act$
    - we take $\alpha = 0.5$
- for a softmax over logits, $\nabla_\theta \log \pi_\theta(\act) = \text{onehot}(\act) - \pi_\theta$
    - for a network, apply the chain rule through the logits


**The update, by numbers** (the figure uses $\rlret = +1$ and $-1$)

<img src="images/policy_gradient_bars.png" width=80%>

- $\rlret = +1$: $\theta \to (0.16, 0.34, -0.51)$ and $\pi_\theta \to (0.37, 0.44, 0.19)$
    - the sampled action gains probability
- $\rlret = -1$: $\theta \to (0.64, -0.34, -0.29)$ and $\pi_\theta \to (0.57, 0.21, 0.22)$
    - the sampled action loses probability
- the other actions move the opposite way (the probabilities sum to $1$)


**Why a baseline? The problem with the raw return**

> **If every return is positive, the raw return pushes every sampled action up: the useful signal is buried in noise.**

- what we want: push an action up if it did *better than usual*, and down if it did *worse*
- the fix: subtract a **baseline** $b$ from the return
    - a baseline is a reference return: what is typical in this state
    - it must not depend on the action taken
    - the weight becomes $\rlret - b$ instead of $\rlret$
    - an action with $\rlret > b$ is pushed up, and one with $\rlret < b$ is pushed down
- subtracting a baseline does *not* change the expected gradient
    - the next slide shows why: the score function has zero mean
    - it changes how noisy the estimate is: its **variance**, how much it fluctuates between samples (as with Monte Carlo targets)
- choosing the baseline is the next session

**A fact we will use again: the zero-mean score lemma**

> **Given everything before step $\tt$, the score $\nabla_\theta \log \pi_\theta(\act \mid \state)$ has mean zero over the action. So a term fixed before the action, times the score, averages to zero.**

$$
\sum_\act \pi_\theta(\act \mid \state) \, \nabla_\theta \log \pi_\theta(\act \mid \state) = \sum_\act \nabla_\theta \pi_\theta(\act \mid \state) = \nabla_\theta \sum_\act \pi_\theta(\act \mid \state) = \nabla_\theta 1 = 0
$$

- the **score function** (the *score*) is $\nabla_\theta \log \pi_\theta(\act \mid \state)$, the gradient of the log probability of the action taken
- **why it matters:** a baseline $b(\state)$ is fixed before the action, so subtracting it leaves the expected gradient unchanged
- **where the lemma returns**
    - B1 appendix: rewards received before an action drop out, so the return-to-go variant equals the whole-episode form
    - B2: a baseline that depends only on the state adds no bias
    - B3: a critic used as a baseline adds no bias
    - C2: a group baseline adds no bias if it does not depend on the completion's own reward
- the proof is in the appendix (recommended)


## From the theorem to an algorithm: sample episodes

**Why this section?** The theorem is an expectation over episodes, and we cannot sum over all episodes. Part A met the same problem and used the same answer: sample.

> **The same move as in Part A: replace an expectation we cannot compute by an average over samples. Here the samples are $N$ episodes, and the result is the estimator of REINFORCE, the simplest policy-gradient method (B2 weight).**

- in Part A, Monte Carlo averaged sampled returns instead of summing over all trajectories
- here we run the current policy $\pi_\theta$ to generate $N$ episodes, and average

$$
\widehat{\nabla_\theta J}(\theta) = \frac{1}{N} \sum_{i=1}^{N} \sum_{\tt=0}^{|\traj_i|-1} \nabla_\theta \log \pi_\theta(\actseq_{i,\tt} \mid \stateseq_{i,\tt}) \, \rlret_{0,\traj_i}
$$

- then take a step $\theta \leftarrow \theta + \alpha \, \widehat{\nabla_\theta J}(\theta)$, where the hat marks the sampled *estimate*
- the environment's effect enters only through the sampled rewards and next states: the model $\transp$ is never needed


## One property of this estimator: it is on-policy

**Why this section?** The estimator just built samples its episodes from the current policy. What does that cost, and how does it connect to the on- and off-policy distinction of Part A?

> **The episodes must come from the policy being improved: the estimator is on-policy.**

- after each step the policy changes, so the old episodes are used up
- contrast deep Q-learning, which reused old transitions through replay because Q-learning is off-policy
- PPO's probability ratio will later let one batch of episodes be reused for several updates (B4 step size)


## Read the proof: the appendix

**Why this section?** The theorem is surprising: the environment drops out. Its proof takes a few slides, and its tools are used next session.

> **We strongly recommend reading the appendix** (`RL_B1_Appendix_PG_proofs.ipynb`) **before the next session: the baseline of the next session is a two-line use of its zero-mean fact.**

- the two ideas in it
    - **the log-derivative trick** (also called the likelihood-ratio trick): $\nabla_\theta p_\theta(x) = p_\theta(x)\, \nabla_\theta \log p_\theta(x)$, which turns the gradient of an expectation into an expectation of a gradient
    - **the log of an episode's probability is a sum of logs**
        - the $\transp$ terms do not depend on $\theta$, so their gradient is zero and they drop out

**What else the appendix holds, and where it is used**

- the appendix also holds
    - why the return-to-go variant equals the episode-return variant (the *score function* has zero mean)
    - an optional second route to the return-to-go variant, from per-step rewards
- **where it is used**
    - the zero-mean fact justifies subtracting a baseline in the next session (the baseline must not depend on the action)
    - later sessions refer back to the log-derivative form $\nabla_\theta \pi_\theta = \pi_\theta \nabla_\theta \log \pi_\theta$
- the main line does not need the proofs to follow the next steps, but without them the theorem is taken on trust


## Preview: one theorem, three slots

**Why this section?** The estimator just built is one formula, and the plan for Part B changes only parts of it. Naming the parts now shows where each later session fits (a plan, to be confirmed as we go).

> **Three slots in one update: how far ($\alpha$), the weight $w$, and, inside the weight, the baseline $b$.**

$$
\theta \leftarrow \theta + \underbrace{\alpha}_{\text{slot 2: how far}} \; \text{average over sampled steps of} \;\; \underbrace{\bigl( \rlret_\tt - \underbrace{b}_{\text{slot 3: baseline source}} \bigr)}_{\text{slot 1: weight } w} \; \nabla_\theta \log \pi_\theta(\act \mid \state)
$$

- in this simplest form the weight is the return minus a baseline $b$
    - from B2 on, the weight $w$ is an advantage $\advseq$, or an estimate of one (a TD error, for example)

<table>
<thead><tr><th nowrap><div align="left">Slot</div></th><th nowrap><div align="left">The question</div></th><th nowrap><div align="left">Planned</div></th></tr></thead>
<tbody>
<tr><td nowrap><div align="left"><b>1. weight</b> $w$: the number that multiplies each step's gradient term (in the theorem: the return)</div></td><td nowrap><div align="left">the raw return is very noisy: what number should multiply each step's gradient term instead? (baseline, advantage; a learned value <i>can</i> help: <i>actor-critic</i>, B3 critic)</div></td><td nowrap><div align="left">B2 weight and B3 critic</div></td></tr>
<tr><td nowrap><div align="left"><b>2. how far</b> one update moves the policy</div></td><td nowrap><div align="left">how far before the estimate stops being trustworthy? (PPO also changes the objective, using a ratio of probabilities)</div></td><td nowrap><div align="left">B4 (step size)</div></td></tr>
<tr><td nowrap><div align="left"><b>3. source of the baseline</b> inside slot 1</div></td><td nowrap><div align="left">can the baseline come without training a value network at all? (GRPO)</div></td><td nowrap><div align="left">Part C</div></td></tr>
</tbody></table>

- the starting point stays fixed (the theorem)
- the methods (REINFORCE, actor-critic, PPO, GRPO) differ in how they fill these slots
- slot 3 sits inside the weight $w$ of slot 1, so the formula shows two places
- the table is a preview now
- it is filled in session by session, and closed in Part C with GRPO
- *actor-critic* is a **use** of a value model (the value-based machinery of Part A) inside this part
- it does not turn Part B into value-based learning


## Aside: methods that learn a policy without a gradient

- Policy iteration (Part A) also improves a policy: it alternates evaluating a value function and improving the policy by an argmax
- so it is both value-based and policy-improving, but it does not move the policy by gradients
- other gradient-free policy search methods exist; this part covers only policy gradient


# What we established, and what it sets up

- **Established**
    - a policy network outputs a distribution (softmax for discrete actions, mean and spread for continuous)
    - the agent samples from it, so exploration is built in
    - $J(\theta)$ is the expected return
    - we climb it by gradient ascent
    - the gradient is an expectation over episodes of $\nabla_\theta \log \pi_\theta$ times a return
        - the transition probabilities $\transp$ do not appear
        - so it can be estimated from sampled episodes
    - the two variants of the theorem (episode return, return-to-go) agree in expectation
    - the raw return is a noisy weight
    - a **baseline** (a reference return) fixes that without changing the expected gradient
    - the estimator is **on-policy**
    - the proofs, including the log-derivative trick, are in the appendix (recommended)
- **Sets up**
    - next session: the raw return is a noisy weight (high variance)
    - baselines and advantages reduce it, and a learned value can supply the baseline (actor-critic)
    - then how far one update may move (PPO), then in Part C a baseline without a critic (GRPO)
